# 2 Link Arm 3D によるRNEA, ABAの検証

モデルを[2 Link Arm](../3D-EoM/2LinkArm/Kane.ipynb)とし、これを対象としたRNEAとABAの計算をPython で From Scratchで記述する。


## 1. モデル定義

<p align="center">
<img src="../3D-EoM/2LinkArm/images/2LinkArmSystem.png" style="width:45%;" />
</p>

リンク形状は直方体とする。<br>
図ではぞれぞれのリンク座標系の基底をR-G-B:$e_x-e_y-e_z$の色で描いている。<br>
$s_1,s_2$はそれぞれの回転関節における回転軸であり、この矢印の右ねじの方向を正方向とする。<br>
$s_1$はリンク1座標系$\Sigma_1$の$e_z$と一致している。$s_2$はリンク2座標系の$e_y$と一致している。

リンクの形状を以下のように$L_x, L_y, L_z$で表し、重心$G_i$において座標系をそのリンクの座標系$\Sigma_i$と平行とする場合。

<p align="center">
<img src="./images/Inertia_shape.png" style="width:15%;" />
</p>

リンク$i$の重心$G_i$における慣性テンソル$I_{G_i}$は以下となる。

$$
{^i}I_{G_i} = \frac{m_i}{12} \begin{bmatrix} {L_{y,i}}^2 + {L_{z,i}}^2 & 0 & 0 \\
0 & {L_{x,i}}^2 + {L_{z,i}}^2 & 0 \\
0 & 0 & {L_{x,i}}^2 + {L_{y,i}}^2
\end{bmatrix}
$$

リンク$i$の原点$O_i$からリンク重心$G_i$までの位置ベクトルを以下とする。

$$
{^i}\boldsymbol{r}_{O_i G_i} = \begin{bmatrix} 0 \\ 0 \\ r_{g_i} \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \\ L_{z,i}/2 \end{bmatrix}
$$

リンク$i$の座標原点$O_i$から子リンク$i+1$の座標原点$O_{i+1}$の位置ベクトルを次のように定義する。

$$
{^i}\boldsymbol{r}_{O_{i} O_{i+1}} = \begin{bmatrix} 0 \\ 0 \\ L_{z,i} \end{bmatrix}
$$

回転行列は以下である。

- ベースからLink1

$$
{^0}R_1 = \begin{bmatrix} \cos(q_1) & -\sin(q_1) & 0 \\ \sin(q_1) & \cos(q_1) & 0 \\ 0 & 0 & 1 \end{bmatrix}
$$

- Link1からLink2
$$
{^1}R_2 = \begin{bmatrix} \cos(q_2) & 0 &\sin(q_2)  \\ 0 & 1 & 0 \\ -\sin(q_2) & 0 & \cos(q_2) \end{bmatrix}
$$

軸の回転を示す Spatial Motion Subspaceは以下である。

- Link1

$$
{^1}\boldsymbol{S}_{1} = \begin{bmatrix} 0 & 0 & 1 & 0 & 0 & 0 \end{bmatrix}^T
$$

- Link2

$$
{^2}\boldsymbol{S}_{2} = \begin{bmatrix} 0 & 1 & 0 & 0 & 0 & 0 \end{bmatrix}^T
$$


## 2. RNEA

RENAでは基準点をそれぞれの座標点原点として計算するため、基準点の情報は省略している。一方、どの座標系表現であるかの情報は記述する。よって、Spatial Vectorの左上の数字は座標系、右下の数字はリンク番号の情報である。

### 2.1 Pass 1 : Forward Recursion

ベースから先端リンクに向かって Spatial Velocity ${^i}\boldsymbol{V}_{i}$, Spatial Acceleartion ${^i}\boldsymbol{A}_{i}$, Spatial Force Vector  ${^i}\boldsymbol{f}_{i}^{\text{net}}$ を次のように順番に計算する。

2リンクアームのベース座標の番号を$0$とする。<br>
ベースは固定ベースとし、重力がWorld座標の$-Z$方向に加わっているとする。

- リンク0 (固定ベース)

$$
\begin{aligned}
{^0}\boldsymbol{V}_{0} &= \begin{bmatrix} {^0}\boldsymbol{\omega}_0^T & {^0}\boldsymbol{v}_0^T \end{bmatrix} = \begin{bmatrix} 0 & 0 & 0 & 0 & 0 & 0 \end{bmatrix}^T \\
{^0}\boldsymbol{A}_{0} &= \begin{bmatrix} {^0}\boldsymbol{\alpha}_0^T & \left({^0}\boldsymbol{a}_0 - {^0}\boldsymbol{G}_{0}\right)^T \end{bmatrix} = \begin{bmatrix} 0 & 0 & 0 & 0 & 0 & 9.81 \end{bmatrix}^T
\end{aligned}
$$

上記の${^0}\boldsymbol{A}_{0}$に重力加速度の補正項を追加した仮想的なBase加速度である。 

- リンク1

$$
\begin{aligned}
{^1}\boldsymbol{V}_{1} &= {^1}X_0 \ {^0}\boldsymbol{V}_{0} + {^1}\boldsymbol{S}_{1} \dot{q}_1 \\
&= {^1}\boldsymbol{S}_{1} \dot{q}_1 \\
{^1}\boldsymbol{A}_{1} &= {^1}X_{0} \ {^0}\boldsymbol{A}_{0} + {^1}\boldsymbol{S}_{1} \ddot{q}_1 + \text{crm}({^1}\boldsymbol{V}_{1}) {^1}\boldsymbol{S}_{1} \dot{q}_1 \\
&= {^1}X_{0} \ {^0}\boldsymbol{A}_{0} + {^1}\boldsymbol{S}_{1} \ddot{q}_1 \\
{^1}\boldsymbol{f}_{1}^{\text{net}} &= {^1}\mathcal{I}_{1} \  {^1}\boldsymbol{A}_{1} + \text{crf}({^1}\boldsymbol{V}_{1}){^1}\mathcal{I}_{1}{^1}\boldsymbol{V}_{1}
\end{aligned}
$$

- リンク2

$$
\begin{aligned}
{^2}\boldsymbol{V}_{2} &= {^2}X_1 \ {^1}\boldsymbol{V}_{1} + {^2}\boldsymbol{S}_{2} \dot{q}_2 \\
{^2}\boldsymbol{A}_{2} &= {^2}X_{1} \ {^1}\boldsymbol{A}_{1} + {^2}\boldsymbol{S}_{2} \ddot{q}_2 + \text{crm}({^2}\boldsymbol{V}_{2}) {^2}\boldsymbol{S}_{2} \dot{q}_2 \\
{^2}\boldsymbol{f}_{2}^{\text{net}} &= {^2}\mathcal{I}_{2} \  {^2}\boldsymbol{A}_{2} + \text{crf}({^2}\boldsymbol{V}_{2}){^2}\mathcal{I}_{2}{^2}\boldsymbol{V}_{2}
\end{aligned}
$$



### 2.2 Pass 2 : Backward Recursion

先端リンクからベースに向かってSpatial Force Vectorを順に計算し、Spatial Motion Subspace によりリンク$i$の関節トルク$\tau_i$を計算する。

- リンク2

$$
\begin{aligned}
{^2}\boldsymbol{f}_{2} &= {^2}\boldsymbol{f}_{2}^{\text{net}} \\
\tau_{2} &= {{^2}\boldsymbol{S}_2}^T \  {^2}\boldsymbol{f}_{2}
\end{aligned}
$$

- リンク1

$$
\begin{aligned}
{^1}\boldsymbol{f}_{1} &= {^1}\boldsymbol{f}_{1}^{\text{net}} + \left( {^2}X_1\right)^T {^2}\boldsymbol{f}_{2} \\
\tau_{1} &= {{^1}\boldsymbol{S}_1}^T \  {^1}\boldsymbol{f}_{1}
\end{aligned}
$$



### 2.3 Spatial Transform, Spatial Inertia, crm, crf

RNEAの再帰計算で必要な $\mathcal{I}, X, \text{crm}, \text{crf}$はそれぞれ次である。

- Spatial Inertia

    リンク$i$座標で表したリンク$i$の原点$O_i$まわりのSpatial Inertia

    $$
    {^i}\mathcal{I}_{i} = 
    \begin{bmatrix}
    {^i}\boldsymbol{I}_{G_i} - m_i [{^i}\boldsymbol{r}_{O_i G_i }]^\land [{^i}\boldsymbol{r}_{O_i G_i}]^\land & m_i [{^i}\boldsymbol{r}_{O_i G_i}]^\land \\
    - m_i [{^i}\boldsymbol{r}_{O_i G_i}]^\land & m_i \boldsymbol{I}_3 
    \end{bmatrix} \in \mathbf{R}^{6 \times 6}
    $$

- Spatial Transform

    リンク$i$から子リンク$i+1$へ座標系を変換し、基準点をリンク$i$の原点$O_i$から子リンク$i+1$の原点$O_{i+1}$へ変換する変換行列

    $$
    {}^{i+1} X_i = \begin{bmatrix}
    {}^{i+1} R_{i} & 0 \\
    -[{}^{i+1}\boldsymbol{r}_{O_i O_{i+1}}]^\land \  {}^{i+1} R_{i} & {}^{i+1} R_{i} 
    \end{bmatrix}
    $$

- Spatial Motion Cross Product

    $$
    \text{crm}({^i}\boldsymbol{V}_i) = \begin{bmatrix} [{^i}\boldsymbol{\omega}_i]^\land & 0 \\ [{^i}\boldsymbol{v}_i]^\land & [{^i}\boldsymbol{\omega}_i]^\land \end{bmatrix}, \quad {^i}\boldsymbol{V}_i = \begin{bmatrix} {^i}\boldsymbol{\omega}_i\\ {^i}\boldsymbol{v}_i\end{bmatrix}
    $$

- Spatial Force Cross Product

    $$
    \text{crf}({^i}\boldsymbol{V}_i) = \begin{bmatrix} [{^i}\boldsymbol{\omega}_i]^\land & [{^i}\boldsymbol{v}_i]^\land \\ 0 & [{^i}\boldsymbol{\omega}_i]^\land \end{bmatrix}, \quad {^i}\boldsymbol{V}_i = \begin{bmatrix} {^i}\boldsymbol{\omega}_i\\ {^i}\boldsymbol{v}_i\end{bmatrix}
    $$

### 2.5 軌道

各リンクの関節をそれぞれ 0 deg から 90 deg 回転させる。それぞれの軌道は5次多項式で計算する。

In [22]:
import numpy as np

def quintic_trajectory(q_A, q_B, T, t):
    s = np.clip(t / T, 0.0, 1.0)
    dq = np.asarray(q_B) - np.asarray(q_A)

    q = q_A + dq * (10*s**3 - 15*s**4 + 6*s**5)
    q_dot = dq / T * (30*s**2 - 60*s**3 + 30*s**4)
    q_ddot = dq / T**2 * (60*s - 180*s**2 + 120*s**3)

    return q, q_dot, q_ddot

In [2]:
# リンク形状(直方体)の慣性テンソル計算関数
# m: 質量
# Lx, Ly, Lz: 各軸方向の長さ
# 戻り値: 3x3の慣性テンソル行列

def calculate_inertia_tensor(m, Lx, Ly, Lz):
    Ixx = (1/12) * m * (Ly**2 + Lz**2)
    Iyy = (1/12) * m * (Lx**2 + Lz**2)
    Izz = (1/12) * m * (Lx**2 + Ly**2)
    return np.array([[Ixx, 0, 0],
                     [0, Iyy, 0],
                     [0, 0, Izz]])

### 2.6 RNEAの実装

In [25]:
import Lib.Spatial_Vector as sv
import Lib.RNEA as rnea

## 計算用パラメータ
number_of_links = 2

### 初期Spatial VelocityとSpatial Acceleration
initial_sp_vel = np.zeros(6)
initial_sp_acc = np.array([0,0,0,0,0,9.81])

### リンクの質量と形状
link_masses = [1.0, 1.0]  # 各リンクの質量
link_lengths = [[0.1, 0.1, 0.5], [0.1, 0.1, 0.5]]  # 各リンクの長さ [Lx, Ly, Lz]

### リンクの重心まわりの慣性テンソル
link_inertia_tensors = [calculate_inertia_tensor(link_masses[i], *link_lengths[i]) for i in range(number_of_links)]

### リンクの原点から重心までのベクトル
link_com_positions = [np.array([0,0,link_lengths[i][2]/2]) for i in range(number_of_links)]

### リンクの原点におけるSpatial Inertia
link_spatial_inertias = [sv.get_spatial_inertia(link_inertia_tensors[i], link_masses[i], link_com_positions[i]) for i in range(number_of_links)]

### リンクi原点から子リンクi+1原点までの距離をリンク座標で表した位置ベクトル
link_positions = np.array([
    [0, 0, 0],
    [0, 0, link_lengths[0][2]],
])

### Spaital Motion Subspace
motion_subspaces = np.array([
    [0,0,1,0,0,0],
    [0,1,0,0,0,0]
])


def calculate_torques_using_RNEA(q_init, q_target, T, t_array):

    forward_pass = rnea.Forward_Pass(number_of_links,initial_sp_vel, initial_sp_acc)
    backward_pass = rnea.Backward_Pass(number_of_links)
    torques = np.zeros((len(t_array), number_of_links))

    ## RNEAによる関節トルク計算
    for t_index, t in enumerate(t_array):
        q, dq, ddq = quintic_trajectory(q_init, q_target, T, t)

        # link1 rotation matrixs
        R_01 = sv.get_Rotation_matrix_on_z(q[0])
        R_12 = sv.get_Rotation_matrix_on_y(q[1])

        X_10 = sv.get_spatial_motion_transform_matrix(R_01.T, R_01.T @ link_positions[0])
        X_21 = sv.get_spatial_motion_transform_matrix(R_12.T, R_12.T @ link_positions[1])

        transform_matrices = [X_10, X_21]


        sp_net_forces = forward_pass.process(omegas=dq, alphas=ddq,
                            transform_matrices=transform_matrices, sbms=motion_subspaces,sp_Is=link_spatial_inertias)
        torques[t_index] = backward_pass.process(transform_matrices=transform_matrices, sbms=motion_subspaces, sp_net_forces=sp_net_forces)

    return torques


In [26]:
## 関節の目標値
q_init = np.deg2rad([0, 90])
q_target = np.deg2rad([0, 90])

## 関節目標値への到達時間
T = 1.0
t_array = np.linspace(0, T, 101)

torques = calculate_torques_using_RNEA(q_init, q_target, T, t_array)
print(torques[-1])

[ 0.     -2.4525]


### 2.7 KANE法の結果を実装

[KANE法による2Link Armの運動方程式](../3D-EoM/2LinkArm/Kane.ipynb)は以下と導出できた。この式を実装し、RNEAとの比較を行う。

$$
M(q)\ddot{q} + c(q, \dot{q}) + g(q) = \tau
$$

$$
\begin{bmatrix}
I_{1zz} +  S_{q_2}^2 (I_{2xx} + m_2r_{2G}^2) + C_{q_2}^2 I_{2zz} & 0 \\
0 & m_2r_{2G}^2 + I_{2yy}
\end{bmatrix}
\begin{bmatrix}
\ddot{q}_1 \\ \ddot{q}_2 
\end{bmatrix} +
\begin{bmatrix}
2S_{q_2}C_{q_2} \left(m_2r_{2G}^2 + I_{2xx} - I_{2zz} \right)\dot{q}_1\dot{q}_2 \\
 S_{q_2}C_{q_2}(I_{2zz}-I_{2xx} - m_2r_{2G}^2)\dot{q}_1^2
\end{bmatrix} +
\begin{bmatrix}
0 \\
- r_{2G} S_{q_2} m_2 g 
\end{bmatrix}=
\begin{bmatrix}
\tau_1 \\ \tau_2
\end{bmatrix}
$$



In [28]:
def calculate_torques_using_KANE(q, dq, ddq, m2, r2G, Ig1, Ig2, g=9.81):

    q2 = q[1]
    dq1, dq2 = dq

    I1zz = Ig1[2,2]
    I2xx = Ig2[0,0]
    I2yy = Ig2[1,1]
    I2zz = Ig2[2,2]

    A = m2 * r2G**2 + I2xx - I2zz

    M = np.array([
        [I1zz + I2zz + A*np.sin(q2)**2, 0],
        [0, I2yy + m2 * r2G**2]
    ])

    c = np.array([
        2*A*np.sin(q2)*np.cos(q2)*dq1*dq2,
        -A*np.sin(q2)*np.cos(q2)*dq1**2
    ])

    g = np.array([
        0,
        -r2G*np.sin(q2)*m2*g
    ])
    
    tau = M @ ddq + c + g
    return tau


target_q = np.deg2rad([0, 90])
calculate_torques_using_KANE(q=target_q, dq=[0,0], ddq=[0,0], m2=link_masses[1], r2G=link_com_positions[1][2], Ig1=link_inertia_tensors[0], Ig2=link_inertia_tensors[1])

    

array([ 0.    , -2.4525])